## Fine-tuning: Conflict type classifier

We fine-tune **DistilBERT** (`distilbert-base-uncased`) as a multi-label classifier on the 5 conflict classes. 
DistilBERT is chosen because our training set is small (~319 snippets) — a lighter model reduces overfitting risk.

**Setup:**
- Task: multi-label classification (a snippet can belong to more than one class simultaneously)
- Loss: BCEWithLogitsLoss (binary cross-entropy per label)
- Split: 80% train / 20% validation
- Metric: macro F1 (treats all 5 classes equally)
- Early stopping patience=3, best checkpoint reloaded at end

After training the model is applied to all cleaned snippets in `snippets_df`.

In [21]:
import re

def clean_snippet(text):
    if pd.isna(text): return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\b\d{6,}\b', '', text)  # remove long number strings
    return text

train_df['snippet_clean'] = train_df['snippet'].apply(clean_snippet)

# filter inference snippets: drop >150 words (boilerplate / garbled)
snippets_df['snippet_clean'] = snippets_df['snippet'].apply(clean_snippet)
snippets_inf = snippets_df[snippets_df['word_count'] <= 150].reset_index(drop=True)

print(f'Training snippets (with conflict label) : {len(train_df)}')
print(f'Inference snippets before filter        : {len(snippets_df)}')
print(f'Inference snippets after >150w filter   : {len(snippets_inf)}')


Training snippets (with conflict label) : 319
Inference snippets before filter        : 83079
Inference snippets after >150w filter   : 82422


In [22]:
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch
from torch.utils.data import Dataset
import numpy as np

MODEL_NAME    = 'distilbert-base-uncased'
MAX_LEN       = 128
CONFLICT_COLS = [f'conflict__{c}' for c in CONFLICT_LABELS]

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ConflictDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc    = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)
        self.labels = labels
    def __len__(self): return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# train / val split
X = train_df['snippet_clean'].tolist()
y = train_df[CONFLICT_COLS].values.tolist()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}  |  Val: {len(X_val)}')

train_ds = ConflictDataset(X_train, y_train)
val_ds   = ConflictDataset(X_val,   y_val)

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CONFLICT_LABELS),
    problem_type='multi_label_classification',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
    }

MODEL_DIR = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_labelling\conflict_model'

args = TrainingArguments(
    output_dir                  = MODEL_DIR,
    num_train_epochs            = 15,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    logging_steps               = 10,
    seed                        = 42,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

# validation report
pred_out  = trainer.predict(val_ds)
probs_val = 1 / (1 + np.exp(-pred_out.predictions))
preds_val = (probs_val >= 0.5).astype(int)
print(classification_report(y_val, preds_val, target_names=CONFLICT_LABELS, zero_division=0))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Train: 255  |  Val: 64


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\charlott\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro
1,0.653500,0.534285,0.000000,0.000000
2,0.494200,0.491567,0.000000,0.000000
3,0.469400,0.456005,0.113889,0.138889
4,0.410800,0.409824,0.357088,0.531915
5,0.360600,0.381045,0.365977,0.556701
6,0.322600,0.350451,0.378818,0.568421
7,0.305100,0.330781,0.359048,0.536082
8,0.280200,0.334495,0.354409,0.534653
9,0.240300,0.305194,0.506691,0.603774
10,0.226100,0.302493,0.568755,0.648649


c:\Users\charlott\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\charlott\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\charlott\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\charlott\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be 

                          precision    recall  f1-score   support

  Environmental & Health       0.81      0.87      0.84        15
         Labour & Social       0.93      0.93      0.93        14
 Governance & Corruption       0.86      0.50      0.63        12
   Economic & Industrial       0.83      0.33      0.48        15
Geopolitical & Strategic       0.43      0.27      0.33        11

               micro avg       0.80      0.60      0.68        67
               macro avg       0.77      0.58      0.64        67
            weighted avg       0.79      0.60      0.66        67
             samples avg       0.62      0.62      0.62        67



## Training results — validation performance

**Train: 255 snippets | Val: 64 snippets** (80/20 split)

The model runs on CPU (no GPU found), which is slower but functionally correct.

### Per-class F1 on validation set

| Conflict class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Environmental & Health | 0.81 | 0.87 | **0.84** | 15 |
| Labour & Social | 0.93 | 0.93 | **0.93** | 14 |
| Governance & Corruption | 0.86 | 0.50 | **0.63** | 12 |
| Economic & Industrial | 0.83 | 0.33 | **0.48** | 15 |
| Geopolitical & Strategic | 0.43 | 0.27 | **0.33** | 11 |

**Macro F1: 0.64 | Micro F1: 0.68**

### Interpretation

**Strong classes:** Labour & Social (F1=0.93) and Environmental & Health (F1=0.84) perform well — these are the two most frequent classes in training and have the most distinctive vocabulary.

**Weak classes:** Economic & Industrial (F1=0.48) and Geopolitical & Strategic (F1=0.33) struggle. Both suffer from low recall, meaning the model is missing many true positives. Likely causes: (1) fewer training examples (46–47 each), and (2) these conflict types use more abstract, policy-level language that is harder to distinguish from each other and from Governance & Corruption.

**Precision is generally higher than recall** — the model is conservative: when it assigns a label it is usually right, but it misses a substantial share of true cases.

### What to do next
- The two strong classes are reliable enough for downstream analysis.
- For the three weaker classes, consider: (1) adding more labeled examples, (2) augmenting training data, or (3) treating their predictions as probabilistic scores rather than hard labels (using  rather than ).
- The  warnings are harmless — they indicate CPU training only; no action needed.

In [23]:
# inference on all cleaned snippets
BATCH_SIZE = 64
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval().to(device)

inf_texts = snippets_inf['snippet_clean'].tolist()
all_probs = []

for i in range(0, len(inf_texts), BATCH_SIZE):
    batch = inf_texts[i : i + BATCH_SIZE]
    enc   = tokenizer(batch, truncation=True, padding=True,
                      max_length=MAX_LEN, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    all_probs.append(torch.sigmoid(logits).cpu().numpy())
    if i % 5000 == 0:
        print(f'  processed {i}/{len(inf_texts)}')

all_probs = np.vstack(all_probs)
all_preds = (all_probs >= 0.5).astype(int)

for j, c in enumerate(CONFLICT_LABELS):
    snippets_inf[f'pred__{c}'] = all_preds[:, j]
    snippets_inf[f'prob__{c}'] = all_probs[:, j].round(3)

snippets_inf['n_conflicts_predicted'] = all_preds.sum(axis=1)

print(f'Snippets with >= 1 conflict predicted : {(snippets_inf["n_conflicts_predicted"] > 0).sum()}')
print(f'Snippets with no conflict predicted   : {(snippets_inf["n_conflicts_predicted"] == 0).sum()}')
print('\nPredicted label distribution:')
for c in CONFLICT_LABELS:
    n = snippets_inf[f'pred__{c}'].sum()
    print(f'  {c}: {n}')

OUT_PATH = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_labelling\snippets_conflict_predictions.csv'
snippets_inf.to_csv(OUT_PATH, index=False)
print(f'\nSaved to: {OUT_PATH}')


  processed 0/82422
  processed 40000/82422


KeyboardInterrupt: 